# Illogical transitions
- This is a comparison within several vector files between years


In [1]:
import os
import geopandas as gpd
import rasterio
from shapely.geometry import LineString, Polygon
from difflib import SequenceMatcher

In [2]:
path=r"Z:\z_resources\im-nca-senegal\v2_shp_occsol_anat\23-12-22\shp_occsol_anat"
illogical_transitions_csv = r""

In [3]:

def get_vector_file_list(path):
    """
    Get a list of the vector files inside the folder
    """
    File_list = [] #f for f in os.listdir(path) if os.isfile(mypath,f)
    for file in os.listdir(path):
        # "anat" is just to get here necessary ones
        if file.endswith("anat.shp"):
            if file not in File_list:
                File_list.append(os.path.join(path,file))
        else:
            pass
    return File_list

In [4]:
def update_names_based_on_similarity(unique_names_gdf1, gdf2, similarity_threshold=0.5):
    """
    Update names in gdf2 based on similarity to names in gdf1.

    Parameters:
    - gdf1 (GeoDataFrame): First GeoDataFrame containing reference names.
    - gdf2 (GeoDataFrame): Second GeoDataFrame whose names need to be updated.
    - similarity_threshold (float): Threshold for similarity ratio. Default is 0.5.

    Returns:
    - None. Updates gdf2 in place.
    """
    # Iterate through rows of gdf2
    for index, row in gdf2.iterrows():
        # Get the value of the "NOM" column for the current row in gdf2
        name_gdf2 = row['NOM']
        highest_similarity_ratio = 0
        best_matching_name = None
        # Iterate through unique names in gdf1
        for name_gdf1 in unique_names_gdf1:
            # Calculate similarity ratio between names in gdf2 and gdf1
            similarity_ratio = SequenceMatcher(None, name_gdf1, name_gdf2).ratio()
            # Update best matching name if similarity ratio is higher
            if similarity_ratio > highest_similarity_ratio:
                highest_similarity_ratio = similarity_ratio
                best_matching_name = name_gdf1

        if similarity_ratio >= similarity_threshold:
            # confirmation = input(f"Similarity found: '{name_gdf2}' -> '{name_gdf1}'Is this okay? (y/n): ").strip().lower()
            # if confirmation == "y":
            gdf2.at[index, 'NOM'] = best_matching_name

    return gdf2

In [5]:
"""We are going to standarize the names of the column NOM since this will be the value which we will """
vector_file_list = get_vector_file_list(path)

names_list = ["Carrière Mine Infrastructure", "Cours d'eau", "Culture irriguée", "Culture maraîchère", "Culture pluviale", "Dune", "Forêt", "Forêt galerie", "Lac", "Localité", "Mangrove", "Mare", "Plaine inondable", "Plantation forestière", "Prairie aquatique", "Savane", "Savane arbustive", "Sol nu", "Steppe", "Tanne", "Vasière"]

gdf1 = gpd.read_file(vector_file_list[0])
gdf2 = gpd.read_file(vector_file_list[1])

# Create a mapping dictionary from gdf1
unique_names_gdf1 = gdf1['NOM'].unique().tolist()
unique_names_original_gdf2 = gdf2['NOM'].unique().tolist()

In [9]:
"""Previous possible geospatial operations"""
def dissolve_and_update_name(gdf, name_to_dissolve, new_name):
    """
    Dissolve polygons with a specific name and update their name.

    Parameters:
    - gdf (GeoDataFrame): Input GeoDataFrame containing polygons.
    - name_to_dissolve (str): Name of the polygons to dissolve.
    - new_name (str): New name for the dissolved polygons.

    Returns:
    - GeoDataFrame: GeoDataFrame with dissolved polygons and updated names.
    """
    # Filter polygons with the specified name
    filtered_polygons = gdf[gdf['NOM'] == name_to_dissolve]
    dissolved_polygon = filtered_polygons.unary_union

    # Create a GeoDataFrame from the dissolved geometry
    dissolved_gdf = gpd.GeoDataFrame(geometry=[dissolved_polygon], crs=gdf.crs)

    # Update the name in the dissolved GeoDataFrame
    dissolved_gdf['NOM'] = new_name

    # Concatenate the dissolved polygons with the rest of the GeoDataFrame
    final_gdf = gpd.GeoDataFrame(pd.concat([gdf[gdf['NOM'] != name_to_dissolve], dissolved_gdf], ignore_index=True), crs=gdf.crs)

    return final_gdf


C:\Users\admin\AppData\Local\Temp\ipykernel_12328\1275880613.py:19: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  gdf1 = gdf1.append({'geometry': merged_polygon, 'NOM': merged_polygon_name}, ignore_index=True)


In [10]:
print(unique_names_gdf1 = gdf1['NOM'].unique().tolist())

TypeError: 'unique_names_gdf1' is an invalid keyword argument for print()

In [6]:
gdf2 = update_names_based_on_similarity(unique_names_gdf1, gdf2, similarity_threshold=0.5)

In [7]:
unique_names_gdf2 = gdf2['NOM'].unique().tolist()
print(unique_names_gdf2)

['Vasière', "Cours d'eau", 'Mare', 'Tanne', 'Carrière Mine Infrastructure', 'Lac', 'Plaine inondable', 'Savane', 'Culture pluviale', 'Mangrove', 'Prairie aquatique', 'Culture irriguée', 'Forêt', 'Culture maraîchère', 'Dune cotière', 'Plantation forestière', 'Sol nu', 'Savane arbustive', 'Forêt galerie', 'Localité', 'Steppe']


In [8]:
print(unique_names_original_gdf2)

['Vasière', "Cours d'eau", 'Mare', 'Tanne', 'Carrière Mine Infrastructure', 'Lac', 'Plaine inondable', 'Savane', 'Culture pluviale', 'Mangrove', 'Prairie aquatique', 'Culture irriguée', 'Forêt', 'Culture maraîchère', 'Dune cotière', 'Plantation forestière', 'Sol nu', 'Savane arbustive', 'Forêt galerie', 'Localité', 'Steppe']


In [ ]:
"""Rasterize the outputs"""

In [17]:
# Perform a spatial join to identify intersecting polygons. This is the most direct think, but it might take a day.
intersections = gpd.sjoin(gdf1, gdf2, how="inner", op='intersects')

c:\ProgramData\Anaconda3\envs\geo_forge_env\lib\site-packages\IPython\core\interactiveshell.py:3382: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):


In [ ]:
def overlay_with_progress(gdf1, gdf2):
    total_operations = len(gdf1) * len(gdf2)
    completed_operations = 0

    for index1, row1 in gdf1.iterrows():
        for index2, row2 in gdf2.iterrows():
            # Perform overlay operation
            new_gdf = gpd.overlay(row1.geometry, row2.geometry, how='union')
            
            # Update progress
            completed_operations += 1
            progress_percent = (completed_operations / total_operations) * 100
            print(f"Progress: {progress_percent:.2f}%")